# 3. Real World Application & Model Uplift
In this notebook, we use a real-world dataset (`statsmodels` Sunspots), inject noise to simulate sensor failures, and demonstrate how our scratch-built filters actively improve downstream Machine Learning forecasting metrics.

In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import sys
import os
sys.path.append(os.path.abspath('..'))
from src.noise_generation import add_spike_noise
from src.filters import median_filter

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:

data = sm.datasets.sunspots.load_pandas().data
df = data[['SUNACTIVITY']].copy()
df.rename(columns={'SUNACTIVITY': 'target'}, inplace=True)

true_signal = df['target'].values


noisy_signal = add_spike_noise(true_signal, num_spikes=int(0.2 * len(true_signal)), spike_amplitude=80)


cleaned_signal = median_filter(noisy_signal, window_size=5)

df['true'] = true_signal
df['noisy'] = noisy_signal
df['cleaned'] = cleaned_signal

plt.figure(figsize=(12, 4))
plt.plot(df['noisy'][:200], label='Noisy Readings', color='lightgrey')
plt.plot(df['true'][:200], label='True Signal', color='black', alpha=0.6)
plt.plot(df['cleaned'][:200], label='Cleaned (Median Filter)', color='red', alpha=0.8)
plt.title('Sunspots Dataset: True vs Noisy vs Cleaned')
plt.legend()
# plt.savefig("/home/ritwik/Pictures/Screenshots/ts_sunspots_output.png")
plt.show()


### Downstream Model Training
We will attempt to forecast the next time step $(y_t)$ using the past 3 steps $(y_{t-1}, y_{t-2}, y_{t-3})$ as features. We will train independent models on the True, Noisy, and Cleaned datasets, and measure RMSE on a hold-out test set.

In [ ]:
def create_features(series, lags=3):
    X, y = [], []
    for i in range(lags, len(series)):
        X.append(series[i-lags:i])
        y.append(series[i])
    return np.array(X), np.array(y)

results = {}
models_preds = {}

for col in ['true', 'noisy', 'cleaned']:
    X, y = create_features(df[col].values)
    
    
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    
    
    model = GradientBoostingRegressor(random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    
    
    _, y_true_all = create_features(df['true'].values)
    y_test_true = y_true_all[split:]
    
    rmse = np.sqrt(mean_squared_error(y_test_true, preds))
    results[col] = rmse
    models_preds[col] = preds

print("Model Forecasting RMSE (Lower is Better)")
for name, score in results.items():
    print(f"{name.capitalize():<10} Data Setting: {score:.2f}")

In [ ]:
plt.figure(figsize=(10,4))

_, y_true_all = create_features(df['true'].values)
y_test_true = y_true_all[split:]

plt.plot(y_test_true[-50:], label='True Ground Truth', color='black', linestyle='--', linewidth=2)
plt.plot(models_preds['noisy'][-50:], label='Forecast (Trained on Noisy)', color='lightgray', alpha=0.8)
plt.plot(models_preds['cleaned'][-50:], label='Forecast (Trained on Cleaned)', color='red', alpha=0.9)

plt.title('Forecasting Uplift: Last 50 Test Points')
plt.legend()
plt.show()